In [ ]:
import torch
from torch import nn, optim
from torch.utils.data import DataLoader
from dataset import FaceDataset,get_transforms
from model import get_xception
import os
DEVICE="cuda" if torch.cuda.is_available() else "cpu"
DATA_DIR="/kaggle/input/fake-face-detection/faces_split"
BATCH_SIZE=16
EPOCHS=10
LR=1e-3
transform=get_transforms()
train_ds=FaceDataset(os.path.join(DATA_DIR,"train"),transform)
train_loader=DataLoader(train_ds,batch_size=BATCH_SIZE,shuffle=True,num_workers=2,pin_memory=True)
model=get_xception()
model.fc =nn.Linear(model.fc.in_features,1)
for param in model.parameters():
    param.requires_grad=False
for param in model.fc.parameters():
    param.requires_grad=True
model=model.to(DEVICE)
criterion=nn.BCEWithLogitsLoss()
optimizer=optim.Adam(model.fc.parameters(),lr=LR)
for epoch in range(EPOCHS):
    model.train()
    total_loss=0

    for imgs,labels in train_loader:
        imgs=imgs.to(DEVICE)
        labels=labels.float().to(DEVICE)
        optimizer.zero_grad()
        outputs=model(imgs).squeeze()
        loss=criterion(outputs,labels)
        loss.backward()
        optimizer.step()
        total_loss+=loss.item()
    print(f"Xception | Epoch {epoch+1}/{EPOCHS} | Loss: {total_loss/len(train_loader):.4f}")
os.makedirs("models", exist_ok=True)
torch.save(model.state_dict(), "models/xception.pth")
print("✅ Xception trained & saved successfully")


Xception | Epoch 1/10 | Loss: 0.3462
Xception | Epoch 2/10 | Loss: 0.3080
Xception | Epoch 3/10 | Loss: 0.3002
Xception | Epoch 4/10 | Loss: 0.2958
Xception | Epoch 5/10 | Loss: 0.2930
Xception | Epoch 6/10 | Loss: 0.2919
Xception | Epoch 7/10 | Loss: 0.2912
Xception | Epoch 8/10 | Loss: 0.2893
Xception | Epoch 9/10 | Loss: 0.2908
Xception | Epoch 10/10 | Loss: 0.2915
✅ Xception trained & saved successfully
